# Case set up
### Import libraries

In [ ]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib as jl
threads = 4

from tqdm import tqdm
from sklearn.model_selection import train_test_split

# tf.config.threading.set_inter_op_parallelism_threads(1)
# tf.config.threading.set_intra_op_parallelism_threads(1)

Visual set up

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib notebook

# Use custom style
style_name = 'paper'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] + '/RESEARCH/styles/matplotlib/' + style_name + '.mplstyle'
) # See: https://matplotlib.org/1.5.3/users/style_sheets.html

In [ ]:
# Get colors list
COLORS = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# Figure/animation saving parameters
# mpl.rcParams['savefig.dpi'] = 100

# Font parameters
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# mpl.rcParams['text.latex.preamble'] = join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Define PrODE and case (model) directories paths

In [ ]:
prode_dir = os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/'
case_dir = "../../ControlONet/trained.2/delta/"

In [ ]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(os.path.normpath(prode_dir))
sys.path.append(os.path.normpath(case_dir))
cwd = os.getcwd()

Change directory

In [ ]:
os.chdir(os.path.abspath(case_dir))

### Import additional libraries

In [ ]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
# Physics-related
from physics.data_generator import DataGenerator
from prode.physics.general import const
from prode.physics.thermochemistry import specie as specieMdl
from prode.physics.thermochemistry.reactionThermo import mixture as mixMdl

### Initialize network

In [ ]:
net = fun._net(
    input_module=inputs
)

### Initialize PrODE model

In [ ]:
from inputs.general import flags

flags['load'] = [f'./saving/delta{i+1}.h5' for i in range(3)]
for k in ('train', 'save'):
    flags[k] = 0

model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)

### Initialize testing data generator
Choose specific test scenarios

In [ ]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = case_dir + "/data/"

Change directory

In [ ]:
os.chdir(os.path.normpath(cwd))

# Postprocessing - Adaptive technique analysis

## Set up analysis

In [ ]:
path = cwd + '/outputs/controller/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'one_pass': 1,
    'accuracy': 0,
    'comp_time': 1
}

In [ ]:
from src.noneqController import NoneqController

# Timescale to grouping strategies
grouping_list = ['RVE1', 'RVE3', 'RVE9', 'RVE27']

# Nonequilibrium distance names
dim = [1, 3, 9]
noneq_dist_names = [
    [r'$\delta^{({%s},{%s})}$' % (i+1, j+1) for j in range(dim[i])] for i in range(len(dim))
]

# Nonequilibrium distance calibration
mep_order = 0
norm_order = 2
mean = True

# x-axis limits
xlim = [1.e-12, 1.e-3]

In [ ]:
def plot_noneq_dist(
    x,
    y,
    x_pred=None,
    y_pred=None,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$\delta$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1.
    )
    # Loop over columns
    n = y.shape[1]
    for d in range(n):
        # >> True values
        ax.plot(
            x,
            y[:,d],
            c=COLORS[d],
            **true_style
        )
        # >> Predicted values
        if y_pred is not None:
            ax.plot(
                x_pred,
                y_pred[:,d],
                label=var_names[d],
                c=COLORS[d],
                **pred_style,
            )
    # Add legend
    if var_names is not None:
        if n > 12:
            fs = 10
        elif n > 6:
            fs = 12
        else:
            fs = 14
        ax.legend(
            [plt.plot([], [], c=COLORS[i])[0] for i in range(len(var_names))],
            var_names,
            fontsize=fs,
            ncol=int(np.ceil(n/5)),
            loc='best'
        )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax
            

Initialize species thermodynamics

In [ ]:
def get_species_thermos(
    grouping_list,
    species
):
    species_thermos = []
    for grouping in grouping_list:
        # Update O2 with the proper grouping strategy
        sp = copy.deepcopy(species)
        sp['O2']['rovib']['grouping'] = grouping
        # Initialize species thermodynamics
        spTh = {
            name: specieMdl.specieThermos.SpecieThermos(
                name,
                info
            ) for name, info in sp.items()
        }
        # Add species thermodynamics to the corresponding timescale
        species_thermos.append(spTh)
    return species_thermos

speciesThermos = get_species_thermos(
    grouping_list,
    inputs.general.species
)

### Custom ICs

In [ ]:
Ttr = [12200., 8000.]
T0 = [2900., 8000.]
P0 = [4.6e3, 7.e3]
XO2 = [0.38, 0.6]

inp = {
    'noneq': [Ttr[0], T0[0], P0[0], XO2[0]],
    'eq': [Ttr[1], T0[1], P0[1], XO2[1]]
}

Generating intial conditions

In [ ]:
if FLAGS['one_pass'] or FLAGS['comp_time']:
    # Generating Initial Conditions
    rho_s = np.zeros((2,dataGen.num_bins+1), dtype=np.float64)
    for i, inp_i in enumerate(inp.values()):
        Ttr, T0, P0, XO2 = inp_i
        XO = 1. - XO2
        nd = P0 / (const.UKB * T0)
        Q_bins = dataGen.partition_func(T0)
        rho_s[i,:-1] = XO2 * Q_bins / sum(Q_bins) * 2.0 * const.mass['O'] * nd
        rho_s[i, -1] = XO  * const.mass['O'] * nd
    rho = np.sum(rho_s, axis=1, keepdims=True)
    Ttr = np.tile(utils.reshape_to_2DArray(Ttr), (2, 1))
    ic = np.hstack((Ttr, rho, np.divide(rho_s, rho)))

    # Evaluate exact solutions
    data_adpt = dataGen.testing_data(ic_test=ic, save=False)
    data_adpt = {k: data_adpt[i] for i, k in enumerate(inp.keys())}

    # Evaluate predicted solutions
    x_adpt, y_adpt, y_pred_adpt = [], [], []
    for k, dat in tqdm(data_adpt.items()):
        x_i, y_i = dat
        x_adpt.append(x_i)
        y_adpt.append(y_i)
        y_pred_adpt.append(model.net.predict(x_i))

### One-pass inference

In [ ]:
if FLAGS['one_pass']:
    path_ = path + '/one_pass/'
    if not os.path.exists(path_):
        os.makedirs(path_)

In [ ]:
if FLAGS['one_pass']:
    utils.print_submain("Plotting nonequilibrium distance")
    for i, name in enumerate(inp.keys()):
        # Plot nonequilibrium distance dynamics
        path_case = path_+'/'+name
        if not os.path.exists(path_case):
            os.makedirs(path_case)
        for j in range(3):
            plot_noneq_dist(
                x=x_adpt[i][1],
                y=y_adpt[i][j],
                x_pred=x_adpt[i][1],
                y_pred=y_pred_adpt[i][j],
                var_names=noneq_dist_names[j],
                path=path_case,
                fig_name='TS'+str(j+1),
                xlim=xlim,
            )

### Accuracy

In [ ]:
if FLAGS['accuracy']:
    path_ = path + '/accuracy/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Num. of evaluation data
eval_pts = 1000

# Time limits
tlim = [1.e-12, 1.e-3]

# Norm order
order = 2

In [ ]:
def mre(order, axis=0, elementwise=False):
    def mean_rel_error(y_true, y_pred):
        if elementwise:
            err = norm((y_pred - y_true) / y_true, order)
            err = np.mean(err, axis=axis)
        else:
            err = np.linalg.norm(y_pred - y_true, ord=order, axis=axis)
            err = err / np.linalg.norm(y_true, ord=order, axis=axis)
        return err
    return mean_rel_error

def norm(x, order):
    x = np.absolute(x)
    if order == 1:
        return x
    else:
        return np.power(np.power(x, order), 1.0 / order)

In [ ]:
if FLAGS['accuracy']:
    utils.print_submain("Evaluating accuracy of the model")
    
    y_pred_, y_ = [], []
    for i in tqdm(range(len(y_pred))):
        mm = [np.argmin(np.absolute(x[i][1]-l)) for l in tlim]
        idx = np.sort(
            np.random.choice(
                np.arange(mm[0], mm[1]+1), eval_pts, replace=False
            )
        )
        y_.append(y[i][-1][idx])
        y_pred_.append(y_pred[i][-1][idx])
    y_pred_ = np.stack(y_pred_, 2)
    y_ = np.stack(y_, 2)

    err = 100*np.mean(
        mre(order, axis=0)(y_, y_pred_), axis=-1, keepdims=True
    )
    
    err_df = pd.DataFrame(
        err.T,
        columns=[f'Y_{i+1}' for i in range(len(err))],
        index=['err']
    )
    err_df = err_df.sort_values(by='err', axis=1, ascending=False)
    err_df.to_csv(path_+'norm.csv')


### Computational time

In [ ]:
if FLAGS['comp_time']:
    path_ = path + '/comp_time/'
    if not os.path.exists(path_):
        os.makedirs(path_)

# Define noneq. metrics tolerances
tol_list = [0, 0.01, 0.1, 0.5]

# Time instants
t_shots = [1.e-9, 1.e-7, 1.e-5, 1.e-3]

# Num. of input data
evaluate = True
iteration = 1

In [ ]:
if FLAGS['comp_time']:
    utils.print_submain("Evaluating computational time")

    if evaluate:
        comp_time = []
        for t in t_shots:
            comp_time_t = []
            for tol in tqdm(tol_list):
                if tol == 0:
                    comp_time_t.append(0.)
                    continue
                comp_time_t_tol = []
                idx = np.argmin(np.absolute(x_adpt[0][1]-t), axis=0)
                x_i = tf.nest.map_structure(lambda x: x[idx,:], x_adpt[0])
#                 yt = tf.nest.map_structure(lambda x: x[idx,:], y_adpt[0])
                for it in range(iteration):
                    start = time.time()
                    y_pred = model.net.call(x_i, tol=tol)
                    end = time.time()
#                     print(tol,  '---', t)
#                     for j in range(3):
#                         yt_j = yt[j] > tol
#                         yp_j = y_pred[j] > tol
#                         print('T:',yt_j)
#                         print('P:',yp_j.numpy())
#                         input('------')
                    nn_time = end - start
                    comp_time_t_tol.append(nn_time)
                comp_time_t.append(np.mean(np.array(comp_time_t_tol)))
            comp_time.append(np.array(comp_time_t))
        comp_time = np.vstack(comp_time).T
        # Save data
        data = np.hstack([np.array(tol_list).reshape(-1,1), comp_time])
        df = pd.DataFrame(
            data,
            columns=['delta']+[f'10^{round(np.log10(t))}' for t in t_shots]
        )
        df.to_csv(path_+'ctrl_time.csv', index=False)